In [ ]:
from dataclasses import dataclass
from huggingface_hub import notebook_login

# Our library stuff
from src.neuronpedia_client import NeuronpediaClient
from src.SAE import JumpReLUSAE

# Visualization
from IPython.display import display, HTML, IFrame
import textwrap

# Disable gradients by default to save memory
torch.set_grad_enabled(False)

# Device setup
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

In [ ]:
@dataclass
class Config:
    """Central configuration for the pipeline."""

    # Model configuration
    model_name: str = "google/gemma-3-27b-it"  # Instruction-tuned Gemma 3 4B

    # SAE configuration
    sae_repo: str = "google/gemma-scope-2-27b-it"
    sae_type: str = "resid_post"  # Type: resid_post, mlp_out, attn_out
    sae_layer: int = 40
    sae_width: str = "65k"  # Width: 16k, 65k, 262k, 1m
    sae_l0: str = "big"

    # Feature extraction
    top_k_features: int = 100  # Number of top features to analyze
    token_position: int = -7
    sample_index_dataset: int = 6

    # BBQ dataset
    bbq_category: str = "Religion"  # Category to analyze
    bbq_condition: str = "disambig"  # ambig or disambig

    # Generation
    max_new_tokens: int = 1024  # Max tokens for CoT generation

    # Neuronpedia
    # Note: Format may need adjustment based on what's available
    neuronpedia_model_id: str = "gemma-3-27b-it"  # May need to verify

    # Parameters for filtering
    max_density = 0.004
    rarity_power = 1.5


    @property
    def sae_path(self) -> str:
        """Construct the SAE path for HuggingFace download."""
        return f"{self.sae_type}/layer_{self.sae_layer}_width_{self.sae_width}_l0_{self.sae_l0}/params.safetensors"

    @property
    def bbq_config(self) -> str:
        """Construct BBQ dataset config name."""
        return f"{self.bbq_category}_{self.bbq_condition}"

# Initialize config
config = Config()

print("Configuration:")
print(f"  Model: {config.model_name}")
print(f"  SAE: {config.sae_repo}")
print(f"  SAE Path: {config.sae_path}")
print(f"  BBQ Config: {config.bbq_config}")


In [ ]:
notebook_login()